# 서울시 60세 이상 문화누리 대상인구 추출

05번의 성·연령·장애별 추정 결과에서 60세 이상만 추출합니다.
성별·연령대와 장애인·비장애인 구분을 유지하며, 인구가 0인 행도 유지합니다.
전체 주민 인구가 아닌 문화누리 대상인구입니다.


## 1. 경로 설정

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = next(
        parent for parent in PROJECT_ROOT.parents
        if (parent / "data").is_dir()
    )

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "estimated_target_population"
INPUT_FILE = DATA_PATH / "서울시_격자_100m_문화누리대상자_성연령장애별_인구수.csv"
OUTPUT_FILE = DATA_PATH / "서울시_격자_100m_문화누리대상자_60세이상_성연령장애별_인구수.csv"

assert INPUT_FILE.exists(), f"입력 파일 없음: {INPUT_FILE}"
assert INPUT_FILE.resolve() != OUTPUT_FILE.resolve()
print(f"입력: {INPUT_FILE}")
print(f"출력: {OUTPUT_FILE}")


입력: C:\Users\ben20\Documents\DataScience\projects\Oracle-Project\data\processed\estimated_target_population\서울시_격자_100m_문화누리대상자_성연령장애별_인구수.csv
출력: C:\Users\ben20\Documents\DataScience\projects\Oracle-Project\data\processed\estimated_target_population\서울시_격자_100m_문화누리대상자_60세이상_성연령장애별_인구수.csv


## 2. 입력 데이터 확인

In [2]:
key_columns = ["GRID_CD", "성별", "연령대"]
population_columns = [
    "문화누리대상자_성연령장애별_추정_인구수",
    "문화누리대상자_성연령비장애별_추정_인구수",
]

population = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
required_columns = key_columns + population_columns
assert set(required_columns).issubset(population.columns), "필수 컬럼 누락"
assert population.isna().sum().sum() == 0, "입력 결측값 확인 필요"
assert not population.duplicated(key_columns).any(), "입력 키 중복"
assert population[population_columns].ge(0).all().all(), "음수 인구 확인 필요"

expected_ages = {
    "0-5세", "6-14세", "15-19세", "20-29세", "30-39세", "40-49세",
    "50-59세", "60-69세", "70-79세", "80-89세", "90-99세", "100세-",
}
assert set(population["연령대"].unique()) == expected_ages, "입력 연령대 정의 변경 확인 필요"
print(f"입력 행 수: {len(population):,}")
print(f"입력 격자 수: {population['GRID_CD'].nunique():,}")
print("연령대:", sorted(population["연령대"].unique()))


입력 행 수: 1,452,672
입력 격자 수: 60,528
연령대: ['0-5세', '100세-', '15-19세', '20-29세', '30-39세', '40-49세', '50-59세', '6-14세', '60-69세', '70-79세', '80-89세', '90-99세']


## 3. 60세 이상 추출 및 품질 확인

60–69세, 70–79세, 80–89세, 90–99세, 100세 이상을 선택합니다.

In [3]:
older_ages = ["60-69세", "70-79세", "80-89세", "90-99세", "100세-"]
older_mask = population["연령대"].isin(older_ages)
population_60_up = population.loc[older_mask].copy()

assert len(population_60_up) > 0, "60세 이상 추출 결과 없음"
assert population_60_up.isna().sum().sum() == 0
assert not population_60_up.duplicated(key_columns).any()
assert set(population_60_up["연령대"].unique()) == set(older_ages)

expected_totals = population.loc[older_mask, population_columns].sum()
assert population_60_up[population_columns].sum().equals(expected_totals)

summary = population_60_up.groupby("연령대")[population_columns].sum().reindex(older_ages)
summary["전체_대상인구"] = summary.sum(axis=1)
display(summary)
print(f"추출 행 수: {len(population_60_up):,}")
print(f"추출 격자 수: {population_60_up['GRID_CD'].nunique():,}")
print(f"60세 이상 대상인구: {int(expected_totals.sum()):,}명")


        문화누리대상자_성연령장애별_추정_인구수  문화누리대상자_성연령비장애별_추정_인구수  전체_대상인구
연령대                                                           
60-69세                   5848                   84334    90182
70-79세                   1087                   54233    55320
80-89세                    257                   22226    22483
90-99세                     18                     745      763
100세-                       0                       0        0
추출 행 수: 605,280
추출 격자 수: 60,528
60세 이상 대상인구: 168,748명


## 4. 결과 저장 및 재확인

In [4]:
population_60_up.to_csv(OUTPUT_FILE, encoding="utf-8-sig", index=False)

saved = pd.read_csv(OUTPUT_FILE, encoding="utf-8-sig")
assert list(saved.columns) == list(population_60_up.columns)
assert len(saved) == len(population_60_up)
assert saved.isna().sum().sum() == 0
assert not saved.duplicated(key_columns).any()
assert saved["연령대"].isin(older_ages).all()
pd.testing.assert_frame_equal(
    saved[key_columns + population_columns],
    population_60_up[key_columns + population_columns].reset_index(drop=True),
)
assert saved[population_columns].sum().equals(expected_totals)
print(f"저장 및 검증 완료: {OUTPUT_FILE.name}")


저장 및 검증 완료: 서울시_격자_100m_문화누리대상자_60세이상_성연령장애별_인구수.csv
